# Bollinger Band Reversal (BBR)

## Import Libs

In [2]:
import pandas as pd 
import numpy as np
import os
from pandas import DataFrame, Series
import plotly.graph_objects as go

## Functions

### Get FE Data

In [3]:
def get_fe_price_data(
        filename: str = "FE_V2_GBPUSD_15mins_1yr_End_20250311.csv"
        ) -> DataFrame:
    """
    Return the FE price data as a DatetimeIndexed 
    DataFrame set to US/Eastern TZ
    """
    FOLDER = "price_data"
    PATH = f"{os.getcwd()}/{FOLDER}"
    df = pd.read_csv(f"{PATH}/{filename}")
    df["Date"] = pd.DatetimeIndex(df["Date"], tz="US/Eastern")
    df.set_index("Date", inplace=True)
    return df

In [4]:
# Set new columns 
gains_cols = [ 
    "Win", "Loss", 
    "TP", "SL", 
    "Gain",
    "Trade_Start", "Trade_End"
    ]

### Simulate Short Positions (Range-Based)

In [2289]:
def range_short_gains(
        df: Series, 
        high: Series, 
        low: Series,
        close: Series,
        signal_name: str,
        sl_pct_range: int,
        tp_pct_range: int,
        bbl: Series,
        atr4: Series,
        sma4_slope: Series,
        range_type: str = "ADR",
        momentum_trade_mgmt = True
        ):
    """Get the pip gain and apply to df"""

    win = 0
    loss = 0
    target_pips = 0 
    sl_pips = 0
    gain = 0
    trade_start =  None
    trade_end = None
    
    if df[signal_name] is True:
        idx = int(df["Idx"])
        iday_idx = int(df["Iday_Idx"])
        # get signal start of fx day timestamp
        day_start_ts = high.iloc[idx-iday_idx:idx-iday_idx+1].index[0]
        OFFSET = day_start_ts + pd.DateOffset(days=1) # end of signal's fx day
        START = df.name # signal start time
        TD = pd.Timedelta(minutes=15)
        END = OFFSET - TD # 17:00 on the signals fx day

        # get stop loss time:
        sl_window = high.loc[START+TD:END] # from signal idx+1 to EOD
        close_window = close.loc[START+TD:END] # from signal idx+1 to EOD
        bbl_window = bbl.loc[START+TD:END]
        atr4_window = atr4.loc[START+TD:END]
        sma4_slope_window = sma4_slope.loc[START+TD:END]
        stop = False # update if stopped 
        sl_ts = None # get timestamp when stopped out
        momentum_stop = False

        for i in range(len(sl_window)): 
            stop_condition = sl_window.iloc[i] >= (df["Close"] + (df[range_type] * sl_pct_range))
            momentum_condition = close_window.iloc[i] < bbl_window.iloc[i] if momentum_trade_mgmt is True else False
            if stop_condition:
                stop = True # trade hit stop loss
                # get stop loss timestamp
                sl_ts = sl_window.iloc[i:i+1].index[0] 
                break
            if momentum_condition:
                momentum_stop = True
                sl_ts = close_window.iloc[i:i+1].index[0]
                break

        # if not stopped out then the stop window is signal:EOD
        # don't include trade if it is the last candle of the day
        if stop is False and momentum_stop is False:
            if close_window.empty is False:
                sl_ts = close_window.iloc[-1:].index[0]
            else:
                sl_ts = START+TD
        # Take profit price
        tp = df["Close"] - (df[range_type] * tp_pct_range)
        
        # trade window
        tp_window = low[START+TD:sl_ts+TD]
        trade_start = START+TD
        trade_end = sl_ts
        target_pips = df["Close"] - tp
        sl_pips = df["Close"] - (df["Close"] + df[range_type] * sl_pct_range) 
        if tp_window.min() <= tp:
            if trade_start == trade_end:
                if stop is False:
                    win = 1
                    gain = df["Close"] - tp
                elif momentum_stop is True:
                    gain = df["Close"] - tp
                    if gain > 0:
                        win = 1
                    else:
                        loss = 1
                else:
                    loss = 1
                    gain = sl_pips
            elif momentum_stop is True:
                gain = df["Close"] - max(tp, close.loc[sl_ts])
                if gain > 0:
                    win = 1
                else:
                    loss = 1
            else:
                win = 1
                gain = df["Close"] - tp                                
        else:
            if stop is True:
                loss = 1
                gain = sl_pips
            elif momentum_stop is True:
                gain = df["Close"] - close.loc[sl_ts]
                if gain > 0:
                    win = 1
                else:
                    loss = 1
            else:
                gain = df["Close"] - close_window.iloc[-1] if close_window.empty is False else 0
                if gain > 0:
                    win = 1
                else:
                    loss = 1
    
    data = win, loss, \
        target_pips, sl_pips, \
        gain, \
        trade_start, trade_end
    
    return data


### Simulate Long Positions (Range-Based)

In [2290]:
def range_long_gains(
        df: Series, 
        high: Series, 
        low: Series,
        close: Series,
        signal_name: str,
        sl_pct_range: int,
        tp_pct_range: int,
        bbu: Series,
        atr4: Series,
        sma4_slope: Series,
        range_type: str = "ADR",
        momentum_trade_mgmt = True
        ):
    """Get the pip gain and apply to df"""

    win = 0
    loss = 0
    target_pips = 0 
    sl_pips = 0
    gain = 0
    trade_start =  None
    trade_end = None
    
    if df[signal_name] is True:
        idx = int(df["Idx"])
        iday_idx = int(df["Iday_Idx"])
        # get signal start of fx day timestamp
        day_start_ts = low.iloc[idx-iday_idx:idx-iday_idx+1].index[0]
        OFFSET = day_start_ts + pd.DateOffset(days=1) # end of signal's fx day
        START = df.name # signal start time
        TD = pd.Timedelta(minutes=15)
        END = OFFSET - TD # 17:00 on the signals fx day

        # get stop loss time:
        sl_window = low.loc[START+TD:END] # from signal idx+1 to EOD
        close_window = close.loc[START+TD:END] # from signal idx+1 to EOD
        bbu_window = bbu.loc[START+TD:END]
        atr4_window = atr4.loc[START+TD:END]
        sma4_slope_window = sma4_slope.loc[START+TD:END]
        stop = False # update if stopped 
        sl_ts = None # get timestamp when stopped out
        momentum_stop = False

        for i in range(len(sl_window)):
            stop_condition = sl_window.iloc[i] <= (df["Close"] - (df[range_type] * sl_pct_range))
            momentum_condition = close_window.iloc[i] > bbu_window.iloc[i] if momentum_trade_mgmt is True else False
            if stop_condition:
                stop = True # trade hit stop loss
                # get stop loss timestamp
                sl_ts = sl_window.iloc[i:i+1].index[0] 
                break
            if momentum_condition:
                momentum_stop = True
                sl_ts = close_window.iloc[i:i+1].index[0]
                break
        # if not stopped out then the stop window is signal:EOD
        # don't include trade if it is the last candle of the day
        if stop is False and momentum_stop is False:
            if close_window.empty is False:
                sl_ts = close_window.iloc[-1:].index[0]
            else:
                sl_ts = START+TD

        # Take profit price
        tp = df["Close"] + (df[range_type] * tp_pct_range)
           
        # trade window
        tp_window = high[START+TD:sl_ts+TD]
        trade_start = START+TD
        trade_end = sl_ts
        target_pips = tp - df["Close"]
        sl_pips = (df["Close"] - df[range_type] * sl_pct_range) - df["Close"] 
        if tp_window.max() >= tp:
            if trade_start == trade_end:
                if stop is False:
                    win = 1
                    gain = tp - df["Close"]
                elif momentum_stop is True:
                    gain = tp - df["Close"]
                    if gain > 0:
                        win = 1
                    else:
                        loss = 1
                else:
                    loss = 1
                    gain = sl_pips
            elif momentum_stop is True:
                gain = min(tp, close.loc[sl_ts]) - df["Close"]
                if gain > 0:
                    win = 1
                else:
                    loss = 1
            else:
                win = 1
                gain = tp - df["Close"]
        else:
            if stop is True:
                loss = 1
                gain = sl_pips
            elif momentum_stop is True:
                gain = close.loc[sl_ts] - df["Close"]
                if gain > 0:
                    win = 1
                else:
                    loss = 1
            else:
                gain = close_window.iloc[-1] - df["Close"] if close_window.empty is False else 0
                if gain > 0:
                    win = 1
                else:
                    loss = 1
    
    data = win, loss, \
        target_pips, sl_pips, \
        gain, \
        trade_start, trade_end, \
        
    return data


### Simulate Limit Short Position

### Simulate Limit Long Position

### Limit-Based Gains (Long/Short)

### Range-Based Gains (Long/Short)

In [4794]:
def get_range_signal_gains(df: DataFrame, long: str, short: str, sl_pct_r, tp_pct_r):
    long_df = df.copy() 
    short_df = df.copy()
    # long
    long_df[gains_cols] = long_df.apply(
        range_long_gains,
        axis=1,
        args=[long_df["High"],long_df["Low"],long_df["Close"],
            long, sl_pct_r, tp_pct_r, long_df["BB_Upper_16_2"], long_df["ATR4"], long_df["SMA4_Slope"]],
        result_type='expand',
        range_type="ATR4",
        momentum_trade_mgmt = True
    )
    # short
    short_df[gains_cols] = short_df.apply(
        range_short_gains,
        axis=1,
        args=[short_df["High"],short_df["Low"],short_df["Close"],
            short, sl_pct_r, tp_pct_r, short_df["BB_Lower_16_2"], short_df["ATR4"], short_df["SMA4_Slope"]],
        result_type='expand',
        range_type="ATR4",
        momentum_trade_mgmt = True
    )
    return pd.concat([long_df, short_df])

### Simulate Long/Short Trend Position

In [5057]:
def trend_gains(
        df: Series, 
        close: Series,
        sma_fast: Series,
        sma_slow: Series,
        long_signal: str,
        short_signal: str
        ):
    """Get the pip gain and apply to df
    
    - long_signal: name of buy signal
    - short_signal: name of sell signal
    """

    win = 0
    loss = 0
    target_pips = 0 
    sl_pips = 0
    gain = 0
    trade_start =  None
    trade_end = None

    if df[long_signal] is True or df[short_signal] is True:
        idx = int(df["Idx"])
        iday_idx = int(df["Iday_Idx"])
        # get signal start of fx day timestamp
        day_start_ts = close.iloc[idx-iday_idx:idx-iday_idx+1].index[0]
        OFFSET = day_start_ts + pd.DateOffset(days=1) # end of signal's fx day
        START = df.name # signal start time
        TD = pd.Timedelta(minutes=15)
        END = OFFSET - TD # 17:00 on the signals fx day

        sma_fast_window = sma_fast.loc[START+TD:END] # from signal idx+1 to EOD
        sma_slow_window = sma_slow.loc[START+TD:END] # from signal idx+1 to EOD
        close_window = close.loc[START+TD:END] # from signal idx+1 to EOD
        stop = False # update if stopped 
        sl_ts = None # get timestamp when stopped out
        
        # find exit timestamp:
        for i in range(len(close_window)): 
            # exit condition
            if df[long_signal] is True:
                exit_condition = 1 if close_window.iloc[i] < sma_fast_window.iloc[i] else 0
            elif df[short_signal] is True:
                exit_condition = 1 if close_window.iloc[i] > sma_fast_window.iloc[i] else 0
            # get stop loss time:
            if exit_condition == 1:
                stop = True # trade hit stop loss
                # get stop loss timestamp
                sl_ts = close_window.iloc[i:i+1].index[0] 
                break
        # if not stopped out then the stop window is signal:EOD
        # don't include trade if it is the last candle of the day
        if stop is False:
            if close_window.empty is False:
                sl_ts = close_window.iloc[-1:].index[0]
            else:
                sl_ts = START+TD if (START+TD) < END else START

        # Win / Loss / Gain / Pips
        if df[long_signal] is True:
            gain = close[sl_ts] - df["Close"]
        elif df[short_signal] is True:
            gain = df["Close"] - close[sl_ts]
        trade_start = START+TD
        trade_end = sl_ts
        target_pips = df["ATR4"]
        sl_pips = -(df["ATR4"])
        win = 1 if gain > 0 else 0
        loss = 1 if gain < 0 else 0
    
    data = win, loss, \
        target_pips, sl_pips, \
        gain, \
        trade_start, trade_end, \
        
    return data



### Trade Stats

In [5325]:
def trade_stats(df: DataFrame, signal_name: str, trade_size: int = 500000, comm: float = 0.00002, quote_ccy: float = 1.34):
    win_count = df.query(f"{signal_name} == True and Win > 0")[f"{signal_name}"].count()
    loss_count = df.query(f"{signal_name} == True and Loss > 0")[f"{signal_name}"].count()
    total_trades = win_count + loss_count
    win_rate = win_count/total_trades * 100
    win = df.query(f"{signal_name} == True and Gain > 0")["Gain"]
    loss = df.query(f"{signal_name} == True and Gain < 0")["Gain"]
    win_avg_pips = win.mean()
    loss_avg_pips = loss.mean()
    win_pips = win.sum()
    loss_pips = loss.sum()
    total_pips = win_pips + loss_pips
    trade_value = trade_size * quote_ccy
    comm = comm * 2 * trade_value

    stats = {
        "Symbol": df["Symbol"].iloc[0],
        "Start": df.index.min(),
        "End": df.index.max(),
        "Win_Count": win_count,
        "Loss_Count": loss_count,
        "Total_Trades": total_trades,
        "Win_Rate": win_rate,
        "Avg_Win": win_avg_pips,
        "Avg_Loss": loss_avg_pips,
        "Win_Pips": win_pips,
        "Loss_Pips": loss_pips,
        "Total_Pips": total_pips,
        "Trade_Value": trade_value,
        "Commissions": comm,
        "Return ($)": round((total_pips * trade_value) - (comm * total_trades),2)
    }

    return stats

def signal_stats(df: DataFrame, signals: list["str"]):
    stats = [trade_stats(df, signal) for signal in signals]
    return stats

def get_trend_signal_stats(df: DataFrame, long: str, short: str, sma_fast: str = "SMA8", sma_slow: str = "SMA16"):
    # Apply gains to df
    df[gains_cols] = df.apply(
        trend_gains,
        axis=1,
        args=[df["Close"], df[sma_fast], df[sma_slow], long, short],
        result_type='expand'
    )

    # get stats for long / short signals
    gain_stats = signal_stats(df, [long,short])

    # build stats dataframe
    return pd.DataFrame(data=gain_stats,
                index=[long,short]
                )

## Price Data Files 

In [5295]:
price_data_files = os.listdir(f"{os.getcwd()}/price_data")
# section filenames by currency pairs
gbp_data = [x for x in price_data_files if "GBPUSD" in x]
gbp_data.sort()
eur_data = [x for x in price_data_files if "EUR" in x]
eur_data.sort()
cad_data = [x for x in price_data_files if "CAD" in x]
cad_data.sort()
jpy_data = [x for x in price_data_files if "JPY" in x]
jpy_data.sort()
aud_data = [x for x in price_data_files if "AUD" in x]
aud_data.sort()
nzd_data = [x for x in price_data_files if "NZD" in x]
nzd_data.sort()
chf_data = [x for x in price_data_files if "CHF" in x]
chf_data.sort()
gbp_aud_data = [x for x in price_data_files if "GBPAUD" in x]
gbp_aud_data.sort()
latest = [x for x in price_data_files if "latest" in x]
# Set max row output to 100 rows
pd.options.display.max_rows = 100

### Feature-Enriched Dataframe Lists

In [ ]:
# get dataframe of feature enriched price data for each year (file)
gbp_df_list = [get_fe_price_data(filename=x) for x in gbp_data]
eur_df_list = [get_fe_price_data(filename=x) for x in eur_data]
cad_df_list = [get_fe_price_data(filename=x) for x in cad_data]
jpy_df_list = [get_fe_price_data(filename=x) for x in jpy_data]
aud_df_list = [get_fe_price_data(filename=x) for x in aud_data]
nzd_df_list = [get_fe_price_data(filename=x) for x in nzd_data]
chf_df_list = [get_fe_price_data(filename=x) for x in chf_data]
latest_df_list = [get_fe_price_data(filename=x) for x in latest]
gbp_aud_df_list = [get_fe_price_data(filename=x) for x in gbp_aud_data]

# Multi-Year Results

## Multi-Year Functions

In [5297]:
def range_based_multi_year_stats(
        df_list: list[DataFrame], 
        long: str, 
        short: str, 
        sl_pct_r: float, 
        tp_pct_r: float,
        get_signal_gains_func: function
        ):
    signal_stats_list = []
    gains_df_list = []
    for df in df_list:
        signal_gains_df: DataFrame = get_signal_gains_func(df, long, short, sl_pct_r, tp_pct_r)
        gains_df_list.append(signal_gains_df)
        long_short_stats = signal_stats(signal_gains_df.between_time("02:00","11:00"), [long,short])
        stats_df = pd.DataFrame(data=long_short_stats, index=[long,short])
        signal_stats_list.append(stats_df)
    return pd.concat(signal_stats_list), pd.concat(gains_df_list)

def trend_based_multi_year_stats(
        df_list: list[DataFrame], 
        long: str, 
        short: str,
        sma: str
        ):
    signal_stats_list = []
    for df in df_list:
        stats_df = get_trend_signal_stats(df, long, short, sma)
        signal_stats_list.append(stats_df)
    return pd.concat(signal_stats_list)

## GBP/USD

### Extreme Momentum

In [5332]:
long_signal = "Bull_XM_V2"
short_signal = "Bear_XM_V2"

stats, gains = range_based_multi_year_stats(gbp_df_list, long_signal, short_signal, 1, 1, get_range_signal_gains)

#### Results (5 Years)

In [5335]:
g = gains
stats["Cum ($)"] = stats["Return ($)"].cumsum().round(2)
stats
# stats["Total_Pips"].sum()
# stats["Win_Count"].mean(), stats["Loss_Count"].mean()
# stats["Win_Rate"].mean()
# stats["Total_Pips"].mean()
# stats["Return ($)"].mean() * 2

,Symbol,Start,End,Win_Count,Loss_Count,Total_Trades,Win_Rate,Avg_Win,Avg_Loss,Win_Pips,Loss_Pips,Total_Pips,Trade_Value,Commissions,Return ($),Cum ($)
Bull_XM_V2,GBPUSD,2021-03-12 02:00:00-05:00,2022-03-11 11:00:00-05:00,173,143,316,54.746835,0.001021,-0.001139,0.176652,-0.162879,0.013774,670000.0,26.8,759.61,759.61
Bear_XM_V2,GBPUSD,2021-03-12 02:00:00-05:00,2022-03-11 11:00:00-05:00,195,132,327,59.633028,0.001272,-0.001317,0.247967,-0.173821,0.074146,670000.0,26.8,40914.39,41674.00
Bull_XM_V2,GBPUSD,2022-03-11 02:00:00-05:00,2023-03-10 11:00:00-05:00,169,148,317,53.312303,0.001856,-0.001979,0.313742,-0.292890,0.020853,670000.0,26.8,5475.58,47149.58
Bear_XM_V2,GBPUSD,2022-03-11 02:00:00-05:00,2023-03-10 11:00:00-05:00,251,198,449,55.902004,0.001768,-0.001847,0.443796,-0.365795,0.078001,670000.0,26.8,40227.64,87377.22
Bull_XM_V2,GBPUSD,2023-03-14 02:00:00-04:00,2024-03-12 11:00:00-04:00,195,172,367,53.133515,0.001104,-0.001132,0.215226,-0.190137,0.025089,670000.0,26.8,6973.86,94351.08
Bear_XM_V2,GBPUSD,2023-03-14 02:00:00-04:00,2024-03-12 11:00:00-04:00,170,148,318,53.459119,0.001230,-0.001353,0.209039,-0.192080,0.016959,670000.0,26.8,2839.96,97191.04
Bull_XM_V2,GBPUSD,2024-03-13 02:00:00-04:00,2025-03-12 11:00:00-04:00,158,140,298,53.020134,0.000911,-0.001188,0.143887,-0.166270,-0.022382,670000.0,26.8,-22982.67,74208.37
Bear_XM_V2,GBPUSD,2024-03-13 02:00:00-04:00,2025-03-12 11:00:00-04:00,133,100,233,57.081545,0.001235,-0.001224,0.164220,-0.122410,0.041810,670000.0,26.8,21768.30,95976.67
Bull_XM_V2,GBPUSD,2025-03-12 02:00:00-04:00,2026-03-11 11:00:00-04:00,186,153,339,54.867257,0.001164,-0.001120,0.216530,-0.171436,0.045094,670000.0,26.8,21127.61,117104.28
Bear_XM_V2,GBPUSD,2025-03-12 02:00:00-04:00,2026-03-11 11:00:00-04:00,171,132,303,56.435644,0.001251,-0.001247,0.213975,-0.164637,0.049337,670000.0,26.8,24935.72,142040.00


#### Trades (Last 10)

In [5338]:
g.loc["2026"][[*gains_cols, "SMA4_Slope", "Bull_XM_V2", "Bear_XM_V2"]].query(
    "Win > 0 or Loss > 0"
).between_time("02:00","11:00").tail(10)

,Win,Loss,TP,SL,Gain,Trade_Start,Trade_End,SMA4_Slope,Bull_XM_V2,Bear_XM_V2
Date,,,,,,,,,,
2026-03-03 06:00:00-05:00,0.0,1.0,0.001799,-0.001799,-0.001799,2026-03-03 06:15:00-05:00,2026-03-03 06:45:00-05:00,-71.916555,False,True
2026-03-03 08:30:00-05:00,1.0,0.0,0.001389,-0.001389,0.001389,2026-03-03 08:45:00-05:00,2026-03-03 11:15:00-05:00,-61.714568,False,True
2026-03-03 08:45:00-05:00,1.0,0.0,0.001317,-0.001317,0.001317,2026-03-03 09:00:00-05:00,2026-03-03 11:15:00-05:00,-72.299572,False,True
2026-03-03 09:00:00-05:00,0.0,1.0,0.001197,-0.001197,-0.001197,2026-03-03 09:15:00-05:00,2026-03-03 09:15:00-05:00,-69.619111,False,True
2026-03-03 09:15:00-05:00,1.0,0.0,0.001494,-0.001494,0.001494,2026-03-03 09:30:00-05:00,2026-03-03 11:15:00-05:00,-59.224785,False,True
2026-03-03 09:30:00-05:00,1.0,0.0,0.001762,-0.001762,0.001762,2026-03-03 09:45:00-05:00,2026-03-03 10:30:00-05:00,-63.950691,False,True
2026-03-03 09:45:00-05:00,1.0,0.0,0.001885,-0.001885,0.001885,2026-03-03 10:00:00-05:00,2026-03-03 10:30:00-05:00,-69.819881,False,True
2026-03-03 10:00:00-05:00,0.0,1.0,0.002221,-0.002221,-0.002221,2026-03-03 10:15:00-05:00,2026-03-03 10:30:00-05:00,-74.628325,False,True
2026-03-11 09:00:00-04:00,0.0,1.0,0.001691,-0.001691,-0.001691,2026-03-11 09:15:00-04:00,2026-03-11 10:00:00-04:00,-74.492972,False,True


### Trend Continuation

In [5339]:
long_signal = "Bull_TC"
short_signal = "Bear_TC"

stats, gains = range_based_multi_year_stats(gbp_df_list, long_signal, short_signal, 1, 1.5, get_range_signal_gains)

#### Results (5 Years)

In [5340]:
g = gains
stats["Cum ($)"] = stats["Return ($)"].cumsum().round(2)
stats

,Symbol,Start,End,Win_Count,Loss_Count,Total_Trades,Win_Rate,Avg_Win,Avg_Loss,Win_Pips,Loss_Pips,Total_Pips,Trade_Value,Commissions,Return ($),Cum ($)
Bull_TC,GBPUSD,2021-03-12 02:00:00-05:00,2022-03-11 11:00:00-05:00,113,152,265,42.641509,0.001550,-0.001113,0.175169,-0.169135,0.006034,670000.0,26.8,-3059.39,-3059.39
Bear_TC,GBPUSD,2021-03-12 02:00:00-05:00,2022-03-11 11:00:00-05:00,121,132,253,47.826087,0.001552,-0.001214,0.187819,-0.160199,0.027621,670000.0,26.8,11725.42,8666.03
Bull_TC,GBPUSD,2022-03-11 02:00:00-05:00,2023-03-10 11:00:00-05:00,129,159,288,44.791667,0.002321,-0.001658,0.299468,-0.263557,0.035911,670000.0,26.8,16341.72,25007.75
Bear_TC,GBPUSD,2022-03-11 02:00:00-05:00,2023-03-10 11:00:00-05:00,147,152,299,49.163880,0.002210,-0.001793,0.324811,-0.272465,0.052346,670000.0,26.8,27058.37,52066.12
Bull_TC,GBPUSD,2023-03-14 02:00:00-04:00,2024-03-12 11:00:00-04:00,115,120,235,48.936170,0.001525,-0.001200,0.175429,-0.144031,0.031398,670000.0,26.8,14738.74,66804.86
Bear_TC,GBPUSD,2023-03-14 02:00:00-04:00,2024-03-12 11:00:00-04:00,104,125,229,45.414847,0.001452,-0.001257,0.150984,-0.155919,-0.004935,670000.0,26.8,-9443.65,57361.21
Bull_TC,GBPUSD,2024-03-13 02:00:00-04:00,2025-03-12 11:00:00-04:00,115,114,229,50.218341,0.001200,-0.000967,0.137997,-0.110284,0.027714,670000.0,26.8,12431.01,69792.22
Bear_TC,GBPUSD,2024-03-13 02:00:00-04:00,2025-03-12 11:00:00-04:00,78,93,171,45.614035,0.001239,-0.001115,0.096628,-0.103726,-0.007098,670000.0,26.8,-9338.54,60453.68
Bull_TC,GBPUSD,2025-03-12 02:00:00-04:00,2026-03-11 11:00:00-04:00,111,127,238,46.638655,0.001568,-0.001144,0.174011,-0.145341,0.028669,670000.0,26.8,12830.08,73283.76
Bear_TC,GBPUSD,2025-03-12 02:00:00-04:00,2026-03-11 11:00:00-04:00,124,116,240,51.666667,0.001412,-0.001084,0.175091,-0.125694,0.049397,670000.0,26.8,26663.91,99947.67


#### Trades (Last 10)

In [5343]:
g[[*gains_cols, "SMA16_Slope", "SMA32_Slope", "Bull_TC", "Bear_TC"]].query(
    "Win > 0 or Loss > 0"
).between_time("02:00","11:00").tail(10)

,Win,Loss,TP,SL,Gain,Trade_Start,Trade_End,SMA16_Slope,SMA32_Slope,Bull_TC,Bear_TC
Date,,,,,,,,,,,
2026-03-03 08:00:00-05:00,1.0,0.0,0.002402,-0.001601,0.002402,2026-03-03 08:15:00-05:00,2026-03-03 11:45:00-05:00,22.518081,-53.941633,False,True
2026-03-03 09:30:00-05:00,1.0,0.0,0.002644,-0.001762,0.002644,2026-03-03 09:45:00-05:00,2026-03-03 10:30:00-05:00,-14.819918,-55.204685,False,True
2026-03-04 10:45:00-05:00,1.0,0.0,0.002430,-0.001620,0.001505,2026-03-04 11:00:00-05:00,2026-03-04 12:15:00-05:00,-22.058201,6.359843,False,True
2026-03-05 02:45:00-05:00,1.0,0.0,0.002076,-0.001384,0.002076,2026-03-05 03:00:00-05:00,2026-03-05 04:00:00-05:00,12.053992,-25.456395,False,True
2026-03-05 03:00:00-05:00,0.0,1.0,0.002539,-0.001692,-0.001692,2026-03-05 03:15:00-05:00,2026-03-05 04:00:00-05:00,-18.863588,-37.812953,False,True
2026-03-05 05:30:00-05:00,0.0,1.0,0.001931,-0.001288,-0.001288,2026-03-05 05:45:00-05:00,2026-03-05 05:45:00-05:00,43.056947,-23.151754,False,True
2026-03-06 07:45:00-05:00,0.0,1.0,0.002151,-0.001434,-0.001434,2026-03-06 08:00:00-05:00,2026-03-06 08:00:00-05:00,-57.876877,-36.078796,False,True
2026-03-10 09:45:00-04:00,0.0,1.0,0.002182,-0.001455,-0.001455,2026-03-10 10:00:00-04:00,2026-03-10 10:30:00-04:00,-22.467133,24.178093,False,True
2026-03-11 10:30:00-04:00,1.0,0.0,0.002799,-0.001866,0.002799,2026-03-11 10:45:00-04:00,2026-03-11 16:45:00-04:00,-31.095229,-28.142141,False,True
